# Blackjack Baseline Workflow

This notebook keeps only the baseline Blackjack flow: setup, training, and evaluation.

It writes training logs under `logs/` and saves the trained PPO model under `models/` so the project outputs are visible and reusable.

## Notebook Layout

1. Import the environment and supporting libraries.
2. Define the baseline environment wrapper and helper functions.
3. Train, save, reload, and evaluate the baseline PPO agent.

In [6]:
from pathlib import Path
import os
import sys

import numpy as np
from gymnasium import ObservationWrapper, spaces
from tqdm.auto import tqdm

# Hide the GPU before Stable-Baselines3 imports torch so behavior stays quiet and reproducible.
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '')

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor

# Make sure the repository root is importable no matter where the notebook is launched from.
ROOT = Path.cwd().resolve()
if not (ROOT / 'blackjack_env.py').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from blackjack_env import BlackjackEnv

c:\Users\tomni\miniconda3\envs\ml-blackjack\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class NoCountObservationWrapper(ObservationWrapper):
    """Expose only the classic Blackjack observation to the agent.

    The environment still tracks running count and true count internally, but
    this wrapper removes those values from the policy input for the baseline run.
    """

    def __init__(self, env):
        super().__init__(env)
        self.observation_space = spaces.Box(
            low=np.array([0.0, 1.0, 0.0], dtype=np.float32),
            high=np.array([31.0, 10.0, 1.0], dtype=np.float32),
            dtype=np.float32,
        )

    def observation(self, observation):
        # Keep only player total, dealer upcard, and usable ace.
        return np.asarray(observation[:3], dtype=np.float32)


def make_env(*, seed=42, monitor_name='baseline'):
    """Create a monitored Blackjack environment for the baseline agent.

    Monitor writes a CSV into `logs/` so the training run leaves a clear artifact.
    """

    env = BlackjackEnv(num_decks=6, penetration=0.75)
    env = NoCountObservationWrapper(env)
    log_dir = ROOT / 'logs' / monitor_name
    log_dir.mkdir(parents=True, exist_ok=True)
    env = Monitor(env, filename=str(log_dir / 'monitor.csv'))
    env.reset(seed=seed)
    return env


def train_agent(env, *, total_timesteps=25_000, seed=42):
    """Train a PPO agent on the supplied environment."""

    model = PPO(
        'MlpPolicy',
        env,
        verbose=1,
        seed=seed,
        n_steps=2048,
        batch_size=64,
        gamma=0.99,
        device='cpu',
    )
    model.learn(total_timesteps=total_timesteps, progress_bar=True)
    return model


def evaluate_agent(model, env, *, episodes=20, seed=123):
    """Evaluate the learned policy over a fixed number of episodes."""

    outcomes = {'wins': 0, 'losses': 0, 'draws': 0}
    rewards = []

    for episode in tqdm(range(episodes), desc='Evaluating baseline agent', leave=False):
        observation, info = env.reset(seed=seed + episode)
        terminated = False
        truncated = False
        last_reward = 0.0

        while not (terminated or truncated):
            action, _ = model.predict(observation, deterministic=True)
            observation, last_reward, terminated, truncated, info = env.step(action)

        reward_value = float(last_reward)
        rewards.append(reward_value)

        if reward_value > 0:
            outcomes['wins'] += 1
        elif reward_value < 0:
            outcomes['losses'] += 1
        else:
            outcomes['draws'] += 1

    return {
        'outcomes': outcomes,
        'mean_reward': float(sum(rewards) / len(rewards)),
    }

SyntaxError: invalid character '】' (U+3011) (1055519396.py, line 21)

## Baseline Training

This section trains the baseline agent, saves the model to `models/`, then reloads it for evaluation so the saved artifact is part of the workflow.

In [3]:
BASELINE_TIMESTEPS = 25_000
BASELINE_MODEL_PATH = ROOT / 'models' / 'blackjack_baseline_ppo'
BASELINE_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

baseline_env = make_env(seed=42, monitor_name='baseline')
baseline_obs, baseline_info = baseline_env.reset(seed=42)
print('Baseline observation space:', baseline_env.observation_space)
print('Baseline sample observation:', baseline_obs)

baseline_model = train_agent(baseline_env, total_timesteps=BASELINE_TIMESTEPS, seed=42)
baseline_model.save(BASELINE_MODEL_PATH)
# Force CPU on load so deserialization does not inherit a CUDA device from earlier sessions.
loaded_baseline_model = PPO.load(BASELINE_MODEL_PATH, env=baseline_env, device='cpu')

baseline_report = evaluate_agent(loaded_baseline_model, baseline_env)
print('Baseline report:')
print(baseline_report)
print(f'Saved baseline model to {BASELINE_MODEL_PATH}')

Baseline observation space: Box([0. 1. 0.], [31. 10.  1.], (3,), float32)
Baseline sample observation: [12.  5.  0.]
Using cpu device
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1.43     |
|    ep_rew_mean     | -0.31    |
| time/              |          |
|    fps             | 1604     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1.28        |
|    ep_rew_mean          | -0.48       |
| time/                   |             |
|    fps                  | 1201        |
|    iterations           | 2           |
|    time_elapsed         | 3           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.016210916 |
|    clip_fraction        | 0